# SatQuery AI — Google Colab Checkpoint Evaluation Pipeline

This notebook performs genuine, automated empirical evaluation of uploaded checkpoints across the official Remote-Sensing benchmarks:
- **RSVQA-LR**: 2,000 held-out Sentinel-2 optical VQA pairs (Overall Accuracy, Presence Accuracy, Comparison Accuracy)
- **VRSBench**: Visual Grounding Box IoU, Precision@0.50, and Token F1 on High-Resolution imagery
- **CDVQA**: Change Detection VQA BLEU-1, BLEU-4, and ROUGE-L across 128 held-out LEVIR-CD scenes
- **Strict Non-Fabrication Policy**: Genuine forward passes executed on GPU; zero synthetic or placeholder scores.


### Step 1: Hardware & GPU Diagnostics
Verify that Colab is allocated a GPU (T4, L4, V100, or A100).

In [ ]:
# Cell 1: Hardware & GPU Diagnostics
import os, sys, torch

print("=" * 60)
print("SatQuery AI — Google Colab Evaluation Environment")
print("=" * 60)
print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model:       {gpu_name}")
    print(f"GPU VRAM:        {total_mem:.2f} GB")
    print(f"CUDA Version:    {torch.version.cuda}")
    print("=" * 60)
    !nvidia-smi
else:
    print("
[WARNING] No GPU detected! Please go to: Runtime -> Change runtime type -> Select T4 GPU or A100.")
    print("=" * 60)


### Step 2: Install Required Dependencies
Installs transformers, accelerate, qwen-vl-utils, peft, and vision libraries.

In [ ]:
# Cell 2: Install Required Dependencies
!pip install --quiet --upgrade \n    "transformers>=4.48.0" \n    "accelerate>=0.26.0" \n    "qwen-vl-utils>=0.0.8" \n    "peft>=0.7.0" \n    "bitsandbytes>=0.43.0" \n    "rasterio>=1.3.0" \n    "timm>=0.9.0" \n    "pillow" \n    "psutil" \n    "pyyaml" \n    "scikit-learn"

print("
Dependencies installed successfully.")


### Step 3: Clone SatQuery Repository & Setup Environment
Clones the feature branch with all benchmarks, evaluators, and manifests.

In [ ]:
# Cell 3: Clone SatQuery Repository
import os, sys

if not os.path.exists("/content/SatQuery"):
    !git clone -b feature/sruthi-single-image https://github.com/Lalith2007/SatQuery.git /content/SatQuery
else:
    %cd /content/SatQuery
    !git fetch origin
    !git checkout feature/sruthi-single-image
    !git pull origin feature/sruthi-single-image

%cd /content/SatQuery

if "/content/SatQuery" not in sys.path:
    sys.path.insert(0, "/content/SatQuery")

print("
SatQuery repository initialized at /content/SatQuery")


### Step 4: Upload & Extract Checkpoint File
Handles .zip, .tar.gz, .safetensors, or .pth uploaded via browser or Google Drive.

In [ ]:
# Cell 4: Upload & Extract Checkpoint
import os, zipfile, tarfile, shutil
from pathlib import Path
from google.colab import files

CHECKPOINT_DIR = Path("/content/checkpoint")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

potential_files = [f for f in os.listdir("/content") if f.endswith((".zip", ".tar.gz", ".tgz", ".tar", ".safetensors", ".pth", ".bin"))]

if potential_files:
    print(f"Found existing checkpoint file in /content: {potential_files[0]}")
    uploaded_filename = potential_files[0]
else:
    print("Please upload your checkpoint file (.zip, .tar.gz, .safetensors, or .pth):")
    uploaded = files.upload()
    uploaded_filename = list(uploaded.keys())[0] if uploaded else None

if uploaded_filename:
    file_path = Path("/content") / uploaded_filename
    print(f"
Processing file: {file_path} ({file_path.stat().st_size / (1024*1024):.2f} MB)")
    
    if uploaded_filename.endswith(".zip"):
        print("Extracting ZIP archive to /content/checkpoint...")
        with zipfile.ZipFile(file_path, "r") as zip_ref:
            zip_ref.extractall(CHECKPOINT_DIR)
    elif uploaded_filename.endswith((".tar.gz", ".tgz", ".tar")):
        print("Extracting TAR archive to /content/checkpoint...")
        with tarfile.open(file_path, "r:*") as tar_ref:
            tar_ref.extractall(CHECKPOINT_DIR)
    else:
        dest = CHECKPOINT_DIR / uploaded_filename
        shutil.copy2(file_path, dest)
        print(f"Copied {uploaded_filename} to {CHECKPOINT_DIR}")

print("
Extracted files in /content/checkpoint:")
for p in CHECKPOINT_DIR.rglob("*"):
    if p.is_file():
        print(f" - {p.relative_to(CHECKPOINT_DIR)} ({p.stat().st_size / (1024*1024):.2f} MB)")


### Step 5: Checkpoint Inspection & Auto-Configuration
Detects LoRA vs Merged model vs Specialist, and ensures adapter_config.json is present.

In [ ]:
# Cell 5: Checkpoint Inspection & Auto-Configuration
import json
from pathlib import Path

CHECKPOINT_DIR = Path("/content/checkpoint")
all_files = list(CHECKPOINT_DIR.rglob("*"))
file_names = [f.name for f in all_files if f.is_file()]

is_lora = any(n in file_names for n in ["adapter_model.safetensors", "adapter_model.bin"])
is_merged = any(n == "config.json" for n in file_names) and any(n.endswith(".safetensors") for n in file_names)
is_tinycd = any("tinycd" in n.lower() or "changedetector" in n.lower() for n in file_names) or any(n.endswith(".pth") and "cmaf" not in n.lower() for n in file_names)
is_cmaf = any("cmaf" in n.lower() for n in file_names)

print("=" * 60)
print("CHECKPOINT INSPECTION REPORT:")
print("=" * 60)
if is_lora:
    print("[DETECTED] Type: Qwen2.5-VL LoRA Adapter")
    adapter_dir = next(f.parent for f in all_files if f.name in ["adapter_model.safetensors", "adapter_model.bin"])
    config_file = adapter_dir / "adapter_config.json"
    if not config_file.exists():
        print("Notice: adapter_config.json missing. Generating matching configuration...")
        default_cfg = {
            "base_model_name_or_path": "Qwen/Qwen2.5-VL-3B-Instruct",
            "bias": "none",
            "fan_in_fan_out": False,
            "lora_alpha": 128,
            "lora_dropout": 0.05,
            "modules_to_save": None,
            "peft_type": "LORA",
            "r": 64,
            "target_modules": ".*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|merger\.linear_fc).*",
            "task_type": "CAUSAL_LM",
            "use_dora": False
        }
        with open(config_file, "w") as f:
            json.dump(default_cfg, f, indent=2)
    print(f"Target Adapter Path: {adapter_dir}")
elif is_merged:
    print("[DETECTED] Type: Qwen2.5-VL Merged Full Model")
    merged_dir = next(f.parent for f in all_files if f.name == "config.json")
    print(f"Target Checkpoint Path: {merged_dir}")
elif is_tinycd:
    print("[DETECTED] Type: TinyCD Bi-Temporal Specialist")
elif is_cmaf:
    print("[DETECTED] Type: CMAF Optical-SAR Specialist")
else:
    print("[DETECTED] Checkpoint files discovered:")
    for f in file_names:
        print(f" - {f}")
print("=" * 60)


### Step 6: Execute Public VLM Benchmark Suite
Runs authentic forward passes across RSVQA-LR, VRSBench, and CDVQA.

In [ ]:
# Cell 6: Execute Public VLM Benchmark Suite
%cd /content/SatQuery

import os
from pathlib import Path

CHECKPOINT_DIR = Path("/content/checkpoint")
OUTPUT_DIR = Path("/content/benchmark_eval_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_files = list(CHECKPOINT_DIR.rglob("*"))
is_adapter = any(f.name in ["adapter_model.safetensors", "adapter_model.bin"] for f in all_files)
is_merged = any(f.name == "config.json" for f in all_files)

if is_adapter:
    adapter_path = next(f.parent for f in all_files if f.name in ["adapter_model.safetensors", "adapter_model.bin"])
    print(f"Running evaluation with adapter: {adapter_path}...")
    !python3 evaluation/post_training_qwen_eval.py \n        --adapter-path "{adapter_path}" \n        --base-model-id "Qwen/Qwen2.5-VL-3B-Instruct" \n        --output-dir "{OUTPUT_DIR}" \n        --device cuda \n        --benchmarks all

elif is_merged:
    merged_path = next(f.parent for f in all_files if f.name == "config.json")
    print(f"Running evaluation with merged model: {merged_path}...")
    !python3 evaluation/post_training_qwen_eval.py \n        --checkpoint-dir "{merged_path}" \n        --output-dir "{OUTPUT_DIR}" \n        --device cuda \n        --benchmarks all

else:
    print("Running validation check (No valid Qwen adapter or merged model found in /content/checkpoint):")
    !python3 evaluation/post_training_qwen_eval.py --output-dir "{OUTPUT_DIR}"


### Step 7: (Optional) Rapid 5-Sample Smoke Test
Run this to verify pipeline health in under 60 seconds before full evaluation.

In [ ]:
# Cell 7: Rapid Smoke Test (5 samples per benchmark)
%cd /content/SatQuery
!python3 evaluation/post_training_qwen_eval.py \n    --adapter-path "/content/checkpoint" \n    --output-dir "/content/smoke_eval_results" \n    --device cuda \n    --max-eval-samples 5


### Step 8: Display Results & Summary Tables
Prints the formatted Markdown results table and JSON summary.

In [ ]:
# Cell 8: Display Formatted Results
from pathlib import Path
import json

md_file = Path("/content/benchmark_eval_results/qwen25vl_benchmark_results.md")
json_file = Path("/content/benchmark_eval_results/qwen25vl_benchmark_results.json")

if md_file.exists():
    print("=" * 60)
    print("EVALUATION RESULTS TABLE:")
    print("=" * 60)
    print(md_file.read_text())

if json_file.exists():
    with open(json_file) as f:
        data = json.load(f)
    print("=" * 60)
    print("EVALUATION JSON METRICS:")
    print("=" * 60)
    print(json.dumps(data, indent=2))


### Step 9: Package and Download Evaluation Artifacts
Bundles the evaluation report files and initiates a browser download.

In [ ]:
# Cell 9: Package and Download Artifacts
from google.colab import files
from pathlib import Path

out_dir = Path("/content/benchmark_eval_results")
archive_name = "/content/satquery_vlm_benchmark_evaluation_results.tar.gz"

if out_dir.exists():
    !tar -czvf "{archive_name}" -C "/content/benchmark_eval_results" .
    print(f"
Created archive: {archive_name}")
    files.download(archive_name)
else:
    print("Output directory /content/benchmark_eval_results does not exist yet.")


### Step 10: (Optional) Specialist Evaluation (TinyCD / CMAF)
Use this if you uploaded a TinyCD or CMAF specialist checkpoint.

In [ ]:
# Cell 10: Specialist Model Evaluation (TinyCD / CMAF)
%cd /content/SatQuery

# TinyCD LEVIR-CD held-out evaluation (128 scenes):
# !python3 specialists/temporal_change/adaptation/run_levir_cd_evaluation.py \n#     --checkpoint "/content/checkpoint/ChangeDetector-TinyCD.pth"

# CMAF WHU-OPT-SAR held-out evaluation (4,950 tiles):
# !python3 specialists/optical_sar/run_whu_opt_sar_evaluation.py \n#     --checkpoint "/content/checkpoint/cmaf_landcover_best.pth"
